# X07 Auditing Language Models for Hidden Objectives


## 目标

把 interpretability、behavior evaluation 和 auditing 连起来，提醒你研究目标不只是“看懂”，还包括找出模型隐藏优化目标。


## 为什么现在学这篇

如果你想把 feature、tracing、editing 和 auditing 连成一条自洽研究线，这篇能把它们放到同一张图里。


## Notebook 与交付

- 原文: https://www.anthropic.com/research/auditing-hidden-objectives
- Notebook：`notebooks/extensions/zh/x07_auditing_hidden_objectives.ipynb`
- 先修: X03, X06, M09
- 在 Colab 里复现一个教学版 auditing toy，并写 1 份 memo，说明如果怀疑模型有隐藏目标，你会先看哪些行为信号、哪些内部证据、哪些停机标准。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

cases = ["harmless", "rewarding", "secret-risk", "high-pressure"]
helpfulness = np.array([0.8, 0.9, 0.5, 0.6])
hidden_objective = np.array([0.1, 0.2, 0.9, 0.8])
behavior_score = 0.7 * helpfulness - 0.4 * hidden_objective
internal_audit_signal = 0.2 * helpfulness + 0.9 * hidden_objective

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(cases, behavior_score, color="#1f5f8b")
axes[0].set_title("Behavior metric")
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(cases, internal_audit_signal, color="#c96a28")
axes[1].set_title("Internal audit signal")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()

flagged = [case for case, signal in zip(cases, internal_audit_signal) if signal > 0.7]
print("Flagged by internal signal:", flagged)
print("Largest behavior drop case:", cases[int(np.argmin(behavior_score))])


## 小结

当行为证据和内部证据指向同一个风险故事时，auditing 才真正形成闭环。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个教学版 auditing toy，并写 1 份 memo，说明如果怀疑模型有隐藏目标，你会先看哪些行为信号、哪些内部证据、哪些停机标准。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 为什么只看行为异常，常常不足以判断是否存在隐藏目标？
- 你的 toy audit 里，哪一种内部证据最能和行为信号形成闭环？
- 如果要把 auditing 工作交给团队，你会把 stop condition 写成什么样？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
